# 🔬 Notebook 3: Code Deployment (CI/CD) — Deep Dive

## 🛠️ Setup

```bash
cd 06-system-designs/code-deployment
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive 1

### Canary deploy with ramp

Canary = send a small % of traffic to the new version. If metrics look good, ramp up; if error rate spikes, roll back.

```
  100% → v1      1% v2 / 99% v1      10% v2 / 90% v1      100% v2
```

In [ ]:
import random

def canary(pct_v2):
    return "v2" if random.random()*100 < pct_v2 else "v1"

random.seed(0)
counts = {"v1":0,"v2":0}
for _ in range(1000):
    counts[canary(10)] += 1
print("10% canary split:", counts)

## Deep dive 2

### SLO-gated rollback

After each ramp step, compare the canary's error-rate to baseline. If worse by more than a threshold, rollback automatically.

In [ ]:
def should_rollback(baseline_err, canary_err, threshold=0.5):
    # rollback if canary errors are >1+threshold× baseline
    return canary_err > baseline_err * (1 + threshold)

# Baseline 0.5% errors; canary 0.6% → still within tolerance
print(should_rollback(0.005, 0.006))
# canary 1.5% → rollback!
print(should_rollback(0.005, 0.015))

## Closing thoughts

- **Artifacts are immutable** — no 'latest' in prod. Always deploy a specific SHA.
- Pipelines are **DAGs**, not sequential scripts — run independent stages in parallel.
- Prefer **canary** to big-bang releases; automate the rollback decision.